# 10 — Inspect Concepts Over Time (descriptive diachronic viewer)

**Does:** for each user-supplied concept, trace it across periods with four complementary, seed-aware lenses: (1) neighbor-overlap stability, (2) axis projection, (3) anchor cosines, (4) frequency trajectory (confound guard). Writes one big reviewable CSV + plots + a plain-language report per concept.

**Reads:** trained `.model` files from `models/word2vec/` (normalizes in RAM; needs no 09 outputs). **Scope:** DESCRIPTIVE ONLY — trajectories with stability bands, no regressions, no p-values, no hypothesis tests.

**How to read movement 'smartly':** a shift is credible only if (a) it replicates across seeds, (b) seed coverage/coherence hold in every period, and (c) frequency cannot explain it. The report prints these checks next to every claim.

In [ ]:
# Cell 1 — CONCEPT INPUT (the only cell you must edit). Concepts supplied at runtime; none hardcoded.
# Single-word mode: TARGETS traces neighbors + anchors. Axis mode: AXIS adds pole-coherence + projection.
CONCEPTS = [
    {"name": "EXAMPLE_replace_me",
     "targets": ["moderator"],
     "pole_a": ["fair", "helpful", "transparent"],
     "pole_b": ["biased", "corrupt", "abusive"],
     "anchors": ["community", "rules"]},
]
SUB_FILTER = None    # e.g. 'AskAcademia'; None = every trained group
K_NEIGHBORS = 20     # top-k for neighbor sets
SEED_FILTER = None   # None = all complete seeds (stability bands need >=2)
print(f"concepts={len(CONCEPTS)} K={K_NEIGHBORS}")

In [ ]:
# Cell 2 — Setup: root, config floors, complete-model index grouped by (group, period) -> seeds.
import os, sys, csv, json, gc, hashlib, datetime
from pathlib import Path
from collections import defaultdict
import yaml
import numpy as np
ROOT = Path("/content/drive/MyDrive/reddit_embeddings_project")
if not ROOT.exists(): ROOT = Path("/home/user/reddit_embeddings_project")
cfg = yaml.safe_load(open(ROOT / "config/project_config.yaml", encoding="utf-8"))
CLAIM_FLOOR = cfg["embeddings"].get("interpretation_floor", 100)
print("config", cfg["config_version"], "claim floor freq =", CLAIM_FLOOR)
import logging
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOGP = ROOT / f"logs/10_inspect__{ts}__cfg-{cfg['config_version']}.log"
LOGP.parent.mkdir(parents=True, exist_ok=True)
lg = logging.getLogger("inspect"); lg.setLevel(logging.INFO); lg.handlers.clear()
fh = logging.FileHandler(LOGP); fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
sh = logging.StreamHandler(sys.stdout); sh.setLevel(logging.WARNING)
lg.addHandler(fh); lg.addHandler(sh)
from gensim.models import Word2Vec
today = datetime.datetime.now(datetime.timezone.utc).date().isoformat()
groups = defaultdict(list)  # (group, period_id) -> [(seed, model_path)]
TMAN = ROOT / "manifests/training_manifest.csv"
if TMAN.exists():
    for r in csv.DictReader(open(TMAN, encoding="utf-8")):
        if r["status"] != "complete": continue
        if SUB_FILTER and r["subreddit_or_group"] != SUB_FILTER: continue
        if SEED_FILTER and int(r["seed"]) not in SEED_FILTER: continue
        mp = Path(r["model_path"])
        if mp.exists() and mp.stat().st_size > 0:
            groups[(r["subreddit_or_group"], r["period_id"])].append((int(r["seed"]), mp))
for k in groups: groups[k].sort()
print(f"(group, period) cells with complete models: {len(groups)}; total models: {sum(len(v) for v in groups.values())}")
if not groups: print("STOP: no complete models — run Notebook 07 first.")
def atomic_text(path: Path, text: str):
    tmp = path.with_suffix(".tmp"); path.parent.mkdir(parents=True, exist_ok=True)
    with open(tmp, "w", encoding="utf-8") as f: f.write(text); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
OUTD = ROOT / "diagnostics/semantic_axes"; OUTD.mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 3 — METRIC ENGINE (per model, in RAM, then released). All vectors L2-normalized first.
# Coverage: present/missing seeds + raw frequencies (claim floor enforced). Coherence: mean pairwise
# cosine within a pole. Separation: mean cross-pole cosine. Axis: mean(a)-mean(b), normalized;
# projection = cosine(target, axis). Jaccard on top-K neighbor sets: within-period across seeds
# (stability) and consecutive-period same seed (change). Frequency logged alongside every metric.
def cos(a, b):
    d = float(np.linalg.norm(a) * np.linalg.norm(b))
    return float(np.dot(a, b) / d) if d else 0.0
def mean_pairwise(vecs):
    if len(vecs) < 2: return None
    s = n = 0.0
    for i in range(len(vecs)):
        for j in range(i + 1, len(vecs)):
            s += cos(vecs[i], vecs[j]); n += 1
    return s / n
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 0.0
def analyze_model(model, concept, K):
    wv = model.wv
    wv.fill_norms()  # normalize once, before any cosine work (brief Stage 10)
    out = {"coverage": {}, "coherence": {}, "freq": {}}
    present = {}
    for pole in ("pole_a", "pole_b"):
        ok, miss = [], []
        for w in concept.get(pole, []) or []:
            if w in wv:
                c = int(wv.get_vecattr(w, "count"))
                out["freq"][w] = c
                (ok if c >= CLAIM_FLOOR else miss).append(w)
            else: miss.append(w)
        present[pole] = ok; out["coverage"][pole] = {"kept": ok, "missing_or_rare": miss}
    for w in (concept.get("targets", []) + concept.get("anchors", [])):
        out["freq"][w] = int(wv.get_vecattr(w, "count")) if w in wv else 0
    va = [wv.get_vector(w, norm=True) for w in present["pole_a"]]
    vb = [wv.get_vector(w, norm=True) for w in present["pole_b"]]
    out["coherence"]["pole_a"] = mean_pairwise(va)
    out["coherence"]["pole_b"] = mean_pairwise(vb)
    out["separation"] = float(np.mean([cos(a, b) for a in va for b in vb])) if va and vb else None
    axis = None
    if va and vb:
        axis = np.mean(va, axis=0) - np.mean(vb, axis=0)
        axis = axis / (np.linalg.norm(axis) or 1.0)
    out["axis"] = axis
    res = {}
    for t in concept.get("targets", []):
        if t not in wv: res[t] = {"in_vocab": False}; continue
        tv = wv.get_vector(t, norm=True)
        nb = [w for w, _ in wv.most_similar(positive=[tv], topn=K)]
        d = {"in_vocab": True, "neighbors": nb,
             "projection": cos(tv, axis) if axis is not None else None,
             "anchors": {a: cos(tv, wv.get_vector(a, norm=True)) for a in concept.get("anchors", []) if a in wv}}
        res[t] = d
    out["targets"] = res
    return out
print("engine ready")

In [ ]:
# Cell 4 — RUN: one model at a time (load, analyze, release). Rows accumulate to the big CSV.
rows = []  # concept, group, period, seed, metric, word, value, aux
cov_summary = defaultdict(dict)
order = sorted(groups)
for (grp, pid) in order:
    for seed, mp in groups[(grp, pid)]:
        try:
            m = Word2Vec.load(str(mp))
        except Exception as e:
            lg.error(f"load {mp}: {e}"); continue
        for c in CONCEPTS:
            try: a = analyze_model(m, c, K_NEIGHBORS)
            except Exception as e: lg.error(f"analyze {c['name']} {pid}: {e}"); continue
            cn = c["name"]
            cov_summary[(cn, grp, pid)][seed] = a["coverage"]
            for pole in ("pole_a", "pole_b"):
                rows.append([cn, grp, pid, seed, "coverage_kept", pole, len(a["coverage"][pole]["kept"]),
                             ";".join(a["coverage"][pole]["kept"])])
                rows.append([cn, grp, pid, seed, "coverage_missing", pole, len(a["coverage"][pole]["missing_or_rare"]),
                             ";".join(a["coverage"][pole]["missing_or_rare"])])
                if a["coherence"][pole] is not None:
                    rows.append([cn, grp, pid, seed, "coherence", pole, round(a["coherence"][pole], 4), ""])
            if a["separation"] is not None:
                rows.append([cn, grp, pid, seed, "separation", "a_vs_b", round(a["separation"], 4), ""])
            for w, f in a["freq"].items():
                rows.append([cn, grp, pid, seed, "frequency", w, f, "below_floor" if 0 < f < CLAIM_FLOOR else ""])
            for t, d in a["targets"].items():
                if not d.get("in_vocab"): rows.append([cn, grp, pid, seed, "target_missing", t, 0, ""]); continue
                if d["projection"] is not None:
                    rows.append([cn, grp, pid, seed, "projection", t, round(d["projection"], 4), ""])
                rows.append([cn, grp, pid, seed, "neighbors", t, 0, ";".join(d["neighbors"])])
                for an, v in d["anchors"].items():
                    rows.append([cn, grp, pid, seed, "anchor_cos", t + "~" + an, round(v, 4), ""])
        del m; gc.collect()
TRAJ = OUTD / "concept_trajectories.csv"
tmp = TRAJ.with_suffix(".tmp")
f = open(tmp, "w", newline="", encoding="utf-8")
w = csv.writer(f); w.writerow(["concept", "group", "period", "seed", "metric", "word", "value", "aux"])
w.writerows(rows); f.flush(); os.fsync(f.fileno()); f.close(); os.replace(tmp, TRAJ)
print(f"trajectories: {len(rows)} rows -> {TRAJ}")

In [ ]:
# Cell 5 — CHANGE VIEW: seed-aware plots. Bands = across-seed spread (stability); lines = per-seed paths.
# Panel A: neighbor Jaccard vs previous period (1.0 = identical neighborhood). Panel B: axis projection
# over time. Panel C: target frequency over time (log) — the confound guardrail (Dubossarsky warning).
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
def series(metric, cn, grp, word):
    d = defaultdict(dict)
    for r in rows:
        if r[0] == cn and r[1] == grp and r[4] == metric and r[5] == word:
            try: d[r[2]][r[3]] = float(r[6])
            except ValueError: pass
    return d
def neighbors_of(cn, grp, pid, seed, t):
    for r in rows:
        if (r[0], r[1], r[2], r[3], r[4], r[5]) == (cn, grp, pid, seed, "neighbors", t):
            return r[7].split(";") if r[7] else []
    return []
made = []
for c in CONCEPTS:
    cn = c["name"]
    for grp in sorted({g for (g, p) in order}):
        pids = sorted({p for (g, p) in order if g == grp})
        if len(pids) < 1: continue
        for t in c.get("targets", []):
            fig, ax = plt.subplots(3, 1, figsize=(9, 10), sharex=True)
            seeds = sorted({s for (g, p) in order if g == grp for (s, _) in groups[(g, p)]})
            for s in seeds:
                js, prev = [], None
                for pid in pids:
                    nb = neighbors_of(cn, grp, pid, s, t)
                    js.append((float(len(set(nb) & set(prev)) / len(set(nb) | set(prev))) if prev is not None and nb and prev else None))
                    prev = nb
                ax[0].plot(pids, js, marker="o", label=f"seed {s}")
            ax[0].set_title(f"{cn} / {t} @{grp}: neighbor overlap vs prev period (Jaccard, K={K_NEIGHBORS})")
            ax[0].set_ylim(-0.05, 1.05); ax[0].legend(fontsize=8); ax[0].grid(True, alpha=0.3)
            pj = series("projection", cn, grp, t)
            for s in seeds:
                ax[1].plot(pids, [pj.get(pid, {}).get(s) for pid in pids], marker="o", label=f"seed {s}")
            ax[1].set_title("axis projection over time (+ = pole_a side)"); ax[1].legend(fontsize=8); ax[1].grid(True, alpha=0.3)
            fq = series("frequency", cn, grp, t)
            for s in seeds[:1]:
                ax[2].plot(pids, [fq.get(pid, {}).get(s) for pid in pids], marker="o", color="black")
            ax[2].axhline(CLAIM_FLOOR, color="red", linestyle="--", label=f"claim floor ({CLAIM_FLOOR})")
            ax[2].set_title("target frequency (confound guardrail)"); ax[2].set_yscale("log")
            ax[2].legend(fontsize=8); ax[2].grid(True, alpha=0.3)
            plt.xticks(rotation=30); fig.tight_layout()
            pp = OUTD / f"traj__{cn}__{str(grp).lower()}__{t}.png"
            fig.savefig(pp, dpi=110); plt.close(fig); made.append(str(pp))
print(f"plots: {len(made)}")
for p in made[:10]: print("  ", p)

In [ ]:
# Cell 6 — PLAIN-LANGUAGE REPORT per concept: what moved, what is fragile, what to check next.
# Warnings fire on: missing/rare seeds, coherence < 0.15, separation >= coherence (poles not distinct),
# single-seed cells (no stability band), target below claim floor. Descriptive only — no tests.
for c in CONCEPTS:
    cn = c["name"]
    L = [f"# Concept report: {cn} ({today}, cfg-{cfg['config_version']})", "",
         "Descriptive only — trajectories with stability bands. No regressions, no p-values.", ""]
    for grp in sorted({g for (g, p) in order}):
        pids = sorted({p for (g, p) in order if g == grp})
        L.append(f"## {grp} ({len(pids)} periods: {', '.join(pids[:6])}{'...' if len(pids) > 6 else ''})")
        for pid in pids:
            cov = cov_summary.get((cn, grp, pid), {})
            if len(cov) < 2:
                L.append(f"- {pid}: WARNING single-seed cell — no stability band; treat as provisional.")
            for pole in ("pole_a", "pole_b"):
                miss = set()
                for s, cv in cov.items(): miss.update(cv.get(pole, {}).get("missing_or_rare", []))
                if miss: L.append(f"- {pid}: {pole} missing/rare: {', '.join(sorted(miss))} (axis for this period excludes them).")
        for t in c.get("targets", []):
            fq = series("frequency", cn, grp, t)
            lo = [p for p in pids for s, v in fq.get(p, {}).items() if v < CLAIM_FLOOR]
            if lo: L.append(f"- {t}: below claim floor ({CLAIM_FLOOR}) in: {', '.join(lo)} — neighbor/projection readings there are fragile.")
        L.append("")
    L.append("## How to claim a shift credibly")
    L.append("1. Same direction in >=2 seeds. 2. Coverage holds every period. 3. Frequency cannot explain it (Panel C flat while A/B move).")
    atomic_text(OUTD / f"report__{cn}.md", "\n".join(L) + "\n")
    print(f"report: report__{cn}.md ({len(L)} lines)")

In [ ]:
# Cell 7 — END-OF-RUN SUMMARY.
print("=" * 70)
print(f"INSPECTION COMPLETE concepts={len(CONCEPTS)} rows={len(rows)} plots={len(made)}")
print(f"big CSV : diagnostics/semantic_axes/concept_trajectories.csv ({len(rows)} rows)")
print("plots   : diagnostics/semantic_axes/traj__<concept>__<group>__<target>.png")
print("reports : diagnostics/semantic_axes/report__<concept>.md")
print("remaining: add concepts (Cell 1) and rerun — completed models are reused, nothing retrains")
print("rerun safe: YES (pure read of the model store; only CSV/plots/reports rewritten atomically)")
print("next     : 11_generate_diagnostics.ipynb (corpus-wide rollup) or back to 07 for more periods")
print("=" * 70)